# Pair Events Into Shifts

Pair each employee's cleaned entry and exit events, including overnight shifts and maximum-gap handling.

**Requires:** `events/events.cleaned.csv`.  
**Produces:** one `*.pairs.csv` file per employee and a pairing report.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "pipeline.json").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / "pipeline.json").exists():
    raise FileNotFoundError("Open this notebook from inside the Nursind repository")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from IPython.display import display
from notebooks import interface as nb
from notebooks.shared_config import load_notebook_context

ctx = load_notebook_context(repo_root / "pipeline.json")
paths = ctx.paths
step_cfg = ctx.step("pair_events")
display(nb.pipeline_overview(ctx, "pair_events"))


## Controls


In [ ]:
VERBOSE = True
MAX_GAP_HOURS = float(step_cfg.get("max_gap_hours", 16.0))
KEEP_INFERRED_COLUMN = bool(step_cfg.get("keep_inferred_column", False))
EMPLOYEE_FILTER = None  # Example: "Mario Rossi"

{
    "max_gap_hours": MAX_GAP_HOURS,
    "keep_inferred_column": KEEP_INFERRED_COLUMN,
    "employee_filter": EMPLOYEE_FILTER,
}


## Input Preview


In [ ]:
display(nb.artifact_table({"cleaned events": paths.cleaned_events_csv}))
display(nb.preview_csv(paths.cleaned_events_csv))


## Build Options


In [ ]:
from core.shifts.pairing.options import PairEmployeeEventsOptions

options = PairEmployeeEventsOptions(
    input_dir=str(paths.events_dir),
    output_dir=str(paths.shifts_dir),
    report_json=str(paths.pairing_report),
    max_gap_hours=MAX_GAP_HOURS,
    employee_filter=EMPLOYEE_FILTER,
    keep_inferred_column=KEEP_INFERRED_COLUMN,
    verbose=VERBOSE,
)
options


## Run Pairing


In [ ]:
from core.drive.logging_utils import setup_logging
from core.shifts.pairing.runtime import run_from_options

setup_logging(VERBOSE)
pair_report = run_from_options(options)
display(nb.report_summary(pair_report))


## Inspect Employee Files


In [ ]:
display(nb.artifact_table({"pairing report": paths.pairing_report}))
display(nb.file_table(paths.shifts_dir, "*.pairs.csv"))
pair_files = sorted(paths.shifts_dir.glob("*.pairs.csv"))
if pair_files:
    display(nb.preview_csv(pair_files[0]))
